# Coco Crepe — 09 Document Unity Catalog

Este notebook documenta las tablas Gold de los tres Data Products mediante comentarios de tabla y columnas en Unity Catalog.

Se ejecuta después de que las tablas Gold hayan sido creadas correctamente.

In [ ]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# El catálogo de inventario se fija manualmente porque coexiste
# con el catálogo usado en la entrega anterior.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

## 1. Definir tablas Gold

In [ ]:
PRODUCT_TABLE = f"{PRODUCT_CATALOG}.gold.product_master_final"
SALES_TABLE = f"{SALES_CATALOG}.gold.sales_summary_final"
INVENTORY_TABLE = f"{INVENTORY_CATALOG}.gold.inventory_status"

print(f"Product table: {PRODUCT_TABLE}")
print(f"Sales table: {SALES_TABLE}")
print(f"Inventory table: {INVENTORY_TABLE}")

## 2. Verificar que las tablas existan

In [ ]:
tables_to_validate = [
    PRODUCT_TABLE,
    SALES_TABLE,
    INVENTORY_TABLE,
]

for table_name in tables_to_validate:
    try:
        spark.table(table_name).limit(1).collect()
        print(f"OK: {table_name}")
    except Exception as exc:
        raise RuntimeError(
            f"No se pudo acceder a la tabla {table_name}. "
            "Verifica que el pipeline haya terminado correctamente."
        ) from exc

## 3. Documentar Product Master

In [ ]:
spark.sql(f"""
COMMENT ON TABLE {PRODUCT_TABLE} IS
'Data Product maestro que publica información confiable, estandarizada y sin duplicados de los productos de Coco Crepe. Es consumido por los Data Products de ventas e inventario.'
""")

product_comments = {
    "product_id": "Identificador único del producto.",
    "product_name": "Nombre comercial del producto, limpio y estandarizado.",
    "category": "Categoría comercial normalizada en mayúsculas.",
    "price": "Precio unitario vigente del producto. Debe ser mayor que cero.",
    "is_active": "Indica si el producto está activo y disponible para otros Data Products.",
    "published_at": "Fecha y hora de publicación del registro en la capa Gold."
}

for column_name, description in product_comments.items():
    escaped_description = description.replace("'", "''")
    spark.sql(f"""
        ALTER TABLE {PRODUCT_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{escaped_description}'
    """)

print("Documentación aplicada a Product Master.")

## 4. Documentar Sales Summary

In [ ]:
spark.sql(f"""
COMMENT ON TABLE {SALES_TABLE} IS
'Data Product que publica indicadores diarios de ventas a partir de órdenes y detalles validados, depurados y deduplicados.'
""")

sales_comments = {
    "order_date": "Fecha de las ventas consolidadas.",
    "total_orders": "Cantidad total de órdenes únicas registradas durante el día.",
    "unique_customers": "Cantidad de clientes únicos que realizaron compras durante el día.",
    "units_sold": "Cantidad total de unidades vendidas durante el día.",
    "total_revenue": "Ingreso total diario calculado a partir del detalle validado de ventas.",
    "average_ticket": "Valor promedio de venta por orden durante el día.",
    "published_at": "Fecha y hora de publicación del Data Product en Gold."
}

for column_name, description in sales_comments.items():
    escaped_description = description.replace("'", "''")
    spark.sql(f"""
        ALTER TABLE {SALES_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{escaped_description}'
    """)

print("Documentación aplicada a Sales Summary.")

## 5. Documentar Inventory Status

In [ ]:
spark.sql(f"""
COMMENT ON TABLE {INVENTORY_TABLE} IS
'Data Product que publica el stock disponible, su clasificación y la valorización del inventario por producto.'
""")

inventory_comments = {
    "product_id": "Identificador único del producto.",
    "product_name": "Nombre estandarizado del producto.",
    "category": "Categoría comercial del producto.",
    "stock": "Cantidad disponible del producto en inventario.",
    "stock_status": "Clasificación del stock: LOW para menos de 50 unidades, MEDIUM entre 50 y 150, y OK para valores superiores.",
    "product_price": "Precio unitario obtenido del Data Product Product Master.",
    "inventory_value": "Valor estimado del inventario, calculado como stock multiplicado por product_price.",
    "supplier_id": "Identificador del proveedor asociado al producto.",
    "supplier_name": "Nombre del proveedor asociado.",
    "supplier_city": "Ciudad del proveedor.",
    "last_update": "Fecha de la última actualización disponible del inventario.",
    "published_at": "Fecha y hora de publicación del Data Product en Gold."
}

for column_name, description in inventory_comments.items():
    escaped_description = description.replace("'", "''")
    spark.sql(f"""
        ALTER TABLE {INVENTORY_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{escaped_description}'
    """)

print("Documentación aplicada a Inventory Status.")

## 6. Validar comentarios en Unity Catalog

In [ ]:
for table_name in [PRODUCT_TABLE, SALES_TABLE, INVENTORY_TABLE]:
    print(f"\nDocumentación de: {table_name}")
    spark.sql(f"DESCRIBE TABLE EXTENDED {table_name}").display()

## Evidencia recomendada

Después de ejecutar este notebook, abre cada tabla en **Catalog Explorer** y toma una captura de:

- la descripción de la tabla;
- la sección **Columns** con los comentarios visibles;
- el nombre completo del catálogo, esquema y tabla.

Estas capturas pueden incluirse en la presentación técnica.